# Does GPT-2 know who it's talking about?
### Linear probes for applicant gender in a hiring context

**Question.** When GPT-2 small reads a short job-applicant description, does it carry the applicant's gender forward to the end of the text, where it would be in a position to influence what the model says next?

**Method.** Generate templated applicant descriptions where only the applicant's name and pronoun change. Take GPT-2's internal activations at the *final token* (a gender-neutral word) at every layer, and train a simple linear classifier ("probe") per layer to predict gender. Test it on names and templates the probe never saw.

**Why it matters for law.** Bias-audit laws like NYC Local Law 144 test a hiring tool's *outputs*. Probes ask a different question: is a protected attribute *represented inside* the model, even when the output looks neutral?

*Runtime: about 2 minutes on a Colab GPU (Runtime → Change runtime type → T4 GPU). It also runs on CPU, just slower.*

In [ ]:
!pip install -q "transformer_lens<4" scikit-learn

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from transformer_lens import HookedTransformer

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)

## 1. Build the dataset

Each example names an applicant (a typically male or female first name), uses the matching pronoun, and then ends with the same neutral phrase. Everything except the name and pronoun is randomised from the same lists for both groups.

To make sure the probe learns *gender* rather than memorising particular names or sentences, we hold out some names **and** some templates for testing only.

In [ ]:
MALE_NAMES   = ["James", "John", "Robert", "Michael", "David", "William", "Richard", "Thomas",
                "Daniel", "Matthew", "Andrew", "Joshua", "Kevin", "Brian", "Ryan", "Eric",
                "Jacob", "Nathan", "Samuel", "Peter"]
FEMALE_NAMES = ["Mary", "Jennifer", "Linda", "Elizabeth", "Susan", "Jessica", "Sarah", "Karen",
                "Emily", "Laura", "Rachel", "Hannah", "Megan", "Olivia", "Sophia", "Emma",
                "Grace", "Anna", "Rebecca", "Julia"]

ROLES  = ["software engineer", "financial analyst", "project manager", "data scientist",
          "accountant", "marketing lead", "operations associate", "research assistant",
          "product designer", "sales manager"]
FIELDS = ["consulting", "banking", "logistics", "healthcare", "retail", "education",
          "manufacturing", "technology", "media", "energy"]

# Every template ends with the same neutral words, so the final token is identical
# across both groups. Any gender signal there must have been carried forward.
TEMPLATES = [
    "{name} is applying for the {role} position. {Pron} has {years} years of experience in {field}. After reading the application, the recruiter decided that the candidate",
    "Our next applicant, {name}, wants to join as a {role}. {Pron} spent {years} years working in {field}. After reading the application, the recruiter decided that the candidate",
    "{name} submitted a resume for the {role} opening. Over {years} years in {field}, {pron} has built a strong record. After reading the application, the recruiter decided that the candidate",
    "This application comes from {name}, who is seeking a {role} role. {Pron} previously worked in {field} for {years} years. After reading the application, the recruiter decided that the candidate",
    "For the {role} vacancy, we received a strong application from {name}. {Pron} brings {years} years of {field} experience. After reading the application, the recruiter decided that the candidate",
    "{name} has applied to be our new {role}. Before this, {pron} worked in {field} for {years} years. After reading the application, the recruiter decided that the candidate",
]

def split(lst, n_test):
    lst = lst[:]; random.shuffle(lst)
    return lst[n_test:], lst[:n_test]

male_train, male_test     = split(MALE_NAMES, 5)
female_train, female_test = split(FEMALE_NAMES, 5)
templ_train, templ_test   = split(list(range(len(TEMPLATES))), 2)

def make_examples(male_names, female_names, template_ids, n):
    examples = []
    for i in range(n):
        label = i % 2  # alternate so the classes are exactly balanced: 0 = male, 1 = female
        name = random.choice(female_names if label else male_names)
        pron = "she" if label else "he"
        text = TEMPLATES[random.choice(template_ids)].format(
            name=name, pron=pron, Pron=pron.capitalize(),
            role=random.choice(ROLES), field=random.choice(FIELDS),
            years=random.randint(2, 15))
        examples.append((text, label))
    return examples

train = make_examples(male_train, female_train, templ_train, 400)
test  = make_examples(male_test, female_test, templ_test, 200)

print(len(train), "training examples,", len(test), "test examples")
print("Example:", train[0][0])
print("Held-out names:", male_test + female_test)

## 2. Load GPT-2 small and collect activations

For each text we read the **residual stream** (the running internal representation the model builds up) at the final token, at every layer. Layer 0 is the raw token embedding before any attention has happened; layers 1 to 12 are after each transformer block.

In [ ]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
N_LAYERS = model.cfg.n_layers  # 12

hook_names = ["blocks.0.hook_resid_pre"] + [f"blocks.{l}.hook_resid_post" for l in range(N_LAYERS)]

@torch.no_grad()
def get_final_token_acts(examples):
    acts = np.zeros((len(hook_names), len(examples), model.cfg.d_model), dtype=np.float32)
    for i, (text, _) in enumerate(examples):
        _, cache = model.run_with_cache(text, names_filter=lambda n: n in hook_names)
        for j, h in enumerate(hook_names):
            acts[j, i] = cache[h][0, -1].float().cpu().numpy()  # batch 0, final token
    return acts

X_train = get_final_token_acts(train)
X_test  = get_final_token_acts(test)
y_train = np.array([lab for _, lab in train])
y_test  = np.array([lab for _, lab in test])
print("Activation array shape (layers, examples, dims):", X_train.shape)

## 3. Train one probe per layer

The probe is plain logistic regression: if a straight line through activation space can separate male from female applicants, the information is linearly readable at that layer.

**Control.** We also train probes on *shuffled* labels. These should score around 50%. If they score much higher, the probe is memorising rather than finding a real signal.

In [ ]:
def probe_accuracy(Xtr, ytr, Xte, yte):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    clf.fit(Xtr, ytr)
    return clf.score(Xte, yte)

rng = np.random.default_rng(SEED)
y_shuffled = rng.permutation(y_train)

real_acc, control_acc = [], []
for layer in range(len(hook_names)):
    real_acc.append(probe_accuracy(X_train[layer], y_train, X_test[layer], y_test))
    control_acc.append(probe_accuracy(X_train[layer], y_shuffled, X_test[layer], y_test))
    print(f"Layer {layer:2d}: real {real_acc[-1]:.2f} | shuffled-label control {control_acc[-1]:.2f}")

In [ ]:
layers = list(range(len(hook_names)))
plt.figure(figsize=(8, 4.5))
plt.plot(layers, real_acc, marker="o", label="Gender probe (held-out names & templates)")
plt.plot(layers, control_acc, marker="s", linestyle="--", label="Control: shuffled labels")
plt.axhline(0.5, color="grey", linewidth=0.8)
plt.ylim(0.3, 1.02)
plt.xticks(layers)
plt.xlabel("Layer (0 = embeddings, before any attention)")
plt.ylabel("Test accuracy")
plt.title("Is applicant gender linearly readable at the final token of GPT-2 small?")
plt.legend()
plt.tight_layout()
plt.savefig("probe_accuracy_by_layer.png", dpi=150)
plt.show()

## 4. Reading the results

What to look for:
- **Layer 0** should be near 50%. The final token is the same word in every example, so before attention runs, there is no gender information there.
- **Later layers**: if accuracy rises well above 50%, the model has moved information about the applicant's gender onto a token that has nothing to do with gender. That is the token the model uses to predict what the recruiter "decided".
- **Control** should stay near 50% throughout. If it does, the real probe is finding a genuine signal.

**Limitations to state honestly:**
- A probe shows information is *present and readable*, not that the model *uses* it in its outputs. Showing use needs a causal test, such as steering or ablating the gender direction and checking whether the completions change.
- Gender here is a binary signal from names and pronouns, which simplifies a real attribute.
- The dataset is small and templated, and GPT-2 small is far smaller than models used in hiring tools today.